In [1]:
import pandas as pd
import numpy as np
import torch

DEVICE = torch.device("cpu")
NUM_GPUS_AVAILABLE = 0

print(f"[INFO] Device yang digunakan: {DEVICE}")
print("[INFO] Menjalankan di CPU (GPU tidak digunakan).")


[INFO] Device yang digunakan: cpu
[INFO] Menjalankan di CPU (GPU tidak digunakan).


In [2]:

# %% ============================================================
# LOAD DATASET
# ============================================================

BASE_DIR = r"C:\Users\s2\Documents\Federated\dataset"

DATA_PATH = (
    f"{BASE_DIR}/drebin-215-dataset-5560malware-9476-benign.csv"
)

data_raw = pd.read_csv(DATA_PATH)

# Copy agar data asli tetap aman
data = data_raw.copy()

print("Ukuran data awal:", data.shape)


# %% ============================================================
# CLEANING DATA
# ============================================================

# Hapus index tambahan jika ada
if "Unnamed: 0" in data.columns:
    data = data.drop(columns=["Unnamed: 0"])


# Kolom khusus
SPECIAL_COLUMN = "TelephonyManager.getSimCountryIso"

if SPECIAL_COLUMN in data.columns:

    # Ganti '?' menjadi NaN
    data[SPECIAL_COLUMN] = data[SPECIAL_COLUMN].replace(
        "?", np.nan
    )

    # Konversi menjadi numerik
    data[SPECIAL_COLUMN] = pd.to_numeric(
        data[SPECIAL_COLUMN],
        errors="coerce"
    )


# Pastikan kolom target tersedia
if "class" not in data.columns:
    raise KeyError(
        "Kolom 'class' tidak ditemukan. "
        f"Kolom tersedia: {list(data.columns)}"
    )


# Mapping label
# B = Benign
# S = Suspicious/Malware
data["class"] = data["class"].replace({
    "B": 0,
    "S": 1
})


# Semua feature selain class
feature_columns = [
    c for c in data.columns
    if c != "class"
]


# Pastikan seluruh feature numerik
data[feature_columns] = data[feature_columns].apply(
    pd.to_numeric,
    errors="coerce"
)


# Hapus missing value
print("\nMissing value sebelum dropna:")
print(data.isna().sum().sum())

data = data.dropna().reset_index(drop=True)

print("Missing value setelah dropna:")
print(data.isna().sum().sum())

print("\nUkuran data setelah cleaning:", data.shape)


Ukuran data awal: (15036, 216)

Missing value sebelum dropna:
5
Missing value setelah dropna:
0

Ukuran data setelah cleaning: (15031, 216)


C:\Users\s2\AppData\Local\Temp\ipykernel_15600\1986145462.py:11: DtypeWarning: Columns (92: TelephonyManager.getSimCountryIso) have mixed types. Specify dtype option on import or set low_memory=False.
  data_raw = pd.read_csv(DATA_PATH)


In [3]:

# %% ============================================================
# DISTRIBUSI LABEL
# ============================================================

print("\nDistribusi class:")

print(data["class"].value_counts())

print("\nDistribusi class (%):")

print(
    data["class"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)



Distribusi class:
class
0    9476
1    5555
Name: count, dtype: int64

Distribusi class (%):
class
0    63.04
1    36.96
Name: proportion, dtype: float64


In [4]:
# %% ============================================================
# KONVERSI DATAFRAME -> NUMPY
# ============================================================

# X = features
X = (
    data
    .drop(columns=["class"])
    .to_numpy(dtype=np.float32)
)

# y = label
y = (
    data["class"]
    .to_numpy(dtype=np.int64)
)


print("\nUkuran dataset:")
print("X:", X.shape)
print("y:", y.shape)

print("\nTipe data:")
print("X:", type(X))
print("y:", type(y))

print("\nDistribusi kelas:")
print(np.bincount(y))


Ukuran dataset:
X: (15031, 215)
y: (15031,)

Tipe data:
X: <class 'numpy.ndarray'>
y: <class 'numpy.ndarray'>

Distribusi kelas:
[9476 5555]


In [5]:
from dataset import prepare_dataset

# 3. Panggil pipeline
data = prepare_dataset(X, y, val_size=0.15, test_size=0.15, task="classification")

X_train_t, y_train_t = data["train"]
X_val_t, y_val_t     = data["val"]
X_test_t, y_test_t   = data["test"]


Train : torch.Size([10521, 215]) torch.Size([10521])
Val   : torch.Size([2255, 215]) torch.Size([2255])
Test  : torch.Size([2255, 215]) torch.Size([2255])


In [6]:
from dataset import prepare_dataset
from model import build_model
from training import train_model
from evaluate import evaluate_model

data = prepare_dataset(X, y, task="classification")
in_dim = data["train"][0].shape[1]
model = build_model(in_dim=in_dim, num_classes=2)

train_model(model, data, epochs=10, verbose=False)

metrics = evaluate_model(model, data["test"],)

Train : torch.Size([10521, 215]) torch.Size([10521])
Val   : torch.Size([2255, 215]) torch.Size([2255])
Test  : torch.Size([2255, 215]) torch.Size([2255])
Accuracy  : 0.9867
Precision : 0.9857
Recall    : 0.9857
F1-score  : 0.9857

Confusion Matrix:
[[1407   15]
 [  15  818]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1422
           1       0.98      0.98      0.98       833

    accuracy                           0.99      2255
   macro avg       0.99      0.99      0.99      2255
weighted avg       0.99      0.99      0.99      2255

0.9866962305986696 0.9857221369560483 0.9857221369560483 0.9857221369560483
